# Multi-Agent RAG POC
This notebook demonstrates the MVP end-to-end flow.

In [ ]:
import sys
import os
# Add root to path so imports work
sys.path.append('..')

from rag.ingest import load_pdf, chunk_text
from rag.embed import embed_texts
from rag.index import VectorStore
from orchestration.graph import app

### 1. Ingest a Sample PDF

In [ ]:
# Create a dummy PDF for testing if one doesn't exist
import fitz
sample_pdf_path = "../data/sample_paper.pdf"

if not os.path.exists(sample_pdf_path):
    doc = fitz.open()
    page = doc.new_page()
    page.insert_text((50, 50), "Machine unlearning in federated learning involves removing a client's contribution without retraining global models from scratch. Several methods exist including decremental learning and exact unlearning.")
    doc.save(sample_pdf_path)
    print("Created dummy PDF")

In [ ]:
pages = load_pdf(sample_pdf_path)
chunks = chunk_text(pages)
print(f"Loaded {len(chunks)} chunks")

### 2. Index Chunks

In [ ]:
chunk_texts = [c['page_content'] for c in chunks]
embeddings = embed_texts(chunk_texts)

vs = VectorStore(persist_directory="../data/chroma_db")
vs.add_documents(chunks, embeddings)
print("Indexed documents")

### 3. Run Agent Workflow

In [ ]:
initial_state = {
    "query": "What are methods for machine unlearning?",
    "expanded_queries": [],
    "retrieved_docs": [],
    "final_response": ""
}

result = app.invoke(initial_state)

print("FINAL OUTPUT:")
print(result['final_response'])